# UABR-SLM · Notebook 02 — BM25 over the 1B-token pretraining subset (not full Wikipedia) + cached top-20 per PopQA question

**Phase 02 of 11** (PLAN §6, §7.2, gate **G2**). CPU only · ~1–2 h · RAM peak ~20 GB · output ~7 GB.

**Inputs (Add Input → Your Work):** notebook **00** output and notebook **01** output.
**Kaggle settings:** Accelerator **None** · Internet **On** (to clone the code) · **Save & Run All**.

**Team decisions implemented (9 Nov 2026):**
- Windows of **60 words, stride 30** (50 % overlap), each **prefixed with its article title**: `"<title>: <60 words>"`.
- **Retrieval corpus = the 1B-token PRETRAINING SUBSET built by notebook 01 — NOT full Wikipedia.**
  It contains every PopQA subject + object article found in the snapshot plus the random fill articles
  (each article once; exposure copies are not indexed). Articles outside that subset (most of the 20231101.en
  snapshot) are **not retrievable**, and distractor windows come only from this subset. This follows the team
  decision "PopQA articles + random fill; same corpus used for retrieval" (PLAN §15 D5). Full-Wikipedia
  retrieval would be a different configuration (~110–125M windows) and is not implemented.
- Query = the PopQA question text. Top-**20** windows are cached for all 14,267 questions, so no later notebook re-runs BM25.
  (Top-2/3 are used as evidence; the rest are available as RAFT distractors.)

**Implementation:** hashed bag of words (lowercase, ≥2-char tokens, English stop words removed, no stemming),
Lucene BM25 (k1 = 0.9, b = 0.4, Pyserini defaults), built in parallel chunks as a sparse matrix so ~25M windows fit in RAM.

**Gate G2 has two parts:**
1. *Automatic* (this notebook): every question gets results; top-3 windows + question fit the 512-token context; answer hit-rates by relation/popularity.
2. *Manual* (the team): label `g2_support_audit.csv` — does each window **state the relation** (not just contain the answer string)? See `G2_AUDIT_README.md`.

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

from uabr.retrieval.bm25 import BM25Config, BM25Index
from uabr.retrieval.eval import answer_in_window, subject_in_window
from uabr.data import tokenizer as T
from uabr.data.wiki_corpus import normalize_title

## 2 · Configuration and inputs

In [ ]:
import numpy as np, pandas as pd
OUT = "/kaggle/working/02_bm25" if ON_KAGGLE else os.path.abspath("outputs/02_bm25")
for d in ["retrieval", "index", "audit"]:
    os.makedirs(os.path.join(OUT, d), exist_ok=True)

bcfg = BM25Config(window_words=60, stride_words=30, prepend_title=True, k1=0.9, b=0.4)
TOPK = 20
SAVE_INDEX = os.environ.get("SAVE_INDEX", "1") == "1"   # ~6 GB; lets later notebooks query new questions
CTX, OVERHEAD, ANSWER_BUDGET = 512, 16, 32               # prompt special tokens/separators + answer allowance

POPQA_PATH = find_input("00_popqa_splits/popqa_prepared.jsonl", "POPQA_PREPARED")
ART_PATH = find_input("01_corpus/corpus/articles.parquet", "ARTICLES")
TOK_PATH = find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER")
STATS01 = json.load(open(find_input("01_corpus/stats.json", "STATS01")))

# What is indexed - stated explicitly, recorded in meta.json, checked in the gate
RETRIEVAL_CORPUS = "pretraining_subset"     # the only implemented option; NOT full Wikipedia
assert RETRIEVAL_CORPUS == "pretraining_subset"
popqa = pd.read_json(POPQA_PATH, lines=True)
assert "in_main" in popqa.columns, "notebook-00 output is v1 - re-run notebook 00"
arts = pd.read_parquet(ART_PATH, columns=["id", "title", "text"])
TITLES, TEXTS = arts["title"].tolist(), arts["text"].tolist()
ART_FINGERPRINT = f"{len(arts)}:{arts['id'].iloc[0]}:{arts['id'].iloc[-1]}"
del arts
tok = T.load(TOK_PATH)
stage(f"PopQA {len(popqa):,} | articles {len(TITLES):,} ({sum(map(len, TEXTS))/1e9:.2f}B chars) | vocab {tok.get_vocab_size()}")
snap = STATS01["snapshot"]
CORPUS_INFO = {
    "retrieval_corpus": RETRIEVAL_CORPUS,
    "description": "notebook-01 pretraining subset: all PopQA subject+object articles found + random fill; each article once",
    "articles_indexed": len(TITLES),
    "of_which_popqa_forced": STATS01["forced_articles"], "of_which_random_fill": STATS01["fill_articles"],
    "natural_tokens": STATS01["natural_tokens"],
    "snapshot_articles": snap["articles"], "snapshot_chars": snap["chars"],
    "share_of_snapshot_articles": len(TITLES) / snap["articles"],
    "share_of_snapshot_chars": sum(map(len, TEXTS)) / snap["chars"],
}
print("RETRIEVAL CORPUS: pretraining subset from notebook 01 (NOT full Wikipedia)")
print(f"  {len(TITLES):,} articles = {CORPUS_INFO['share_of_snapshot_articles']:.1%} of the snapshot's "
      f"{snap['articles']:,} articles ({CORPUS_INFO['share_of_snapshot_chars']:.1%} of its text); "
      f"{STATS01['forced_articles']:,} PopQA-forced + {STATS01['fill_articles']:,} random fill")
bcfg.to_dict()

## 3 · Build windows + BM25 index

In [ ]:
try:
    import psutil; ram = lambda: f"RAM used {psutil.Process().memory_info().rss/1e9:.1f} GB"
except ImportError:
    ram = lambda: ""
ix = BM25Index(bcfg).build(TITLES, TEXTS, log=stage)
N_WIN = ix.W.shape[0]
stage(f"{N_WIN:,} windows from {len(TITLES):,} articles ({N_WIN/len(TITLES):.1f}/article) {ram()}")

## 4 · Window length in tokens

In [ ]:
rng = np.random.default_rng(0)
sample_ids = rng.choice(N_WIN, size=min(20000, N_WIN), replace=False)
win_tok = np.array([len(a) for a in T.encode_batch(tok, [ix.window(i, TITLES, TEXTS) for i in sample_ids])])
print("tokens per window (sample of 20k): " + ", ".join(f"p{q}={np.percentile(win_tok, q):.0f}" for q in [5, 50, 95, 99]))

## 5 · Retrieve top-20 for all PopQA questions

In [ ]:
n = len(popqa)
doc_ids = np.full((n, TOPK), -1, np.int64); scores = np.zeros((n, TOPK), np.float32)
for i, q in enumerate(popqa["question"]):
    d, s = ix.search(q, TOPK)
    doc_ids[i, :len(d)] = d; scores[i, :len(s)] = s
    if (i + 1) % 2000 == 0:
        stage(f"  retrieved {i+1:,}/{n:,}")
n_short = int((doc_ids < 0).any(axis=1).sum())
stage(f"done; questions with < {TOPK} results: {n_short}")

## 6 · Automatic diagnostics: answer hit, subject match, support proxy

In [ ]:
uniq = np.unique(doc_ids[doc_ids >= 0])
WTEXT = {int(d): ix.window(int(d), TITLES, TEXTS) for d in uniq}
WTITLE = {int(d): TITLES[int(ix.doc_article[d])] for d in uniq}

ans = np.zeros((n, TOPK), bool); subj = np.zeros((n, TOPK), bool); own = np.zeros((n, TOPK), bool)
for i, r in enumerate(popqa.itertuples(index=False)):
    st = normalize_title(r.s_wiki_title)
    names = [r.subj] + list(r.s_aliases)
    for j, d in enumerate(doc_ids[i]):
        if d < 0: continue
        w = WTEXT[int(d)]
        ans[i, j] = answer_in_window(w, r.gold_aliases)
        own[i, j] = normalize_title(WTITLE[int(d)]) == st
        subj[i, j] = subject_in_window(w, WTITLE[int(d)], st, names)
sup = ans & subj

def at(k, M): return M[:, :k].any(axis=1)
KS = [1, 2, 3, 5, 10, 20]
popqa["answer_hit@3"], popqa["support_proxy@3"], popqa["own_article@3"] = at(3, ans), at(3, sup), at(3, own)
main = popqa["in_main"].to_numpy()

hit_by_split = pd.DataFrame({f"{name}@{k}": pd.Series(at(k, M)[main]).groupby(popqa.loc[main, "split"].to_numpy()).mean()
                             for k in KS for name, M in [("answer_hit", ans), ("support_proxy", sup)]}).round(3)
by_rel = popqa[main].groupby("prop")[["answer_hit@3", "support_proxy@3", "own_article@3"]].mean().round(3)
by_pop = popqa[main].groupby("pop_bin")[["answer_hit@3", "support_proxy@3", "own_article@3"]].mean().round(3)
hit_by_split.to_csv(f"{OUT}/retrieval/hit_by_split.csv"); by_rel.to_csv(f"{OUT}/retrieval/hit_by_relation.csv")
by_pop.to_csv(f"{OUT}/retrieval/hit_by_popbin.csv")
print("in_main questions | support_proxy = answer alias AND subject (own article or name) in the same window")
display(hit_by_split); display(by_rel.sort_values("answer_hit@3")); display(by_pop)

## 7 · Context budget: does question + top-k windows fit in 512 tokens?

In [ ]:
q_tok = np.array([len(a) for a in T.encode_batch(tok, popqa["question"].tolist())])
wt = {d: len(a) for d, a in zip(WTEXT, T.encode_batch(tok, list(WTEXT.values())))}
fit = {}
for k in [2, 3]:
    tot = q_tok + np.array([sum(wt[int(d)] for d in row[:k] if d >= 0) for row in doc_ids]) + OVERHEAD + ANSWER_BUDGET
    fit[k] = {"fit_rate": float((tot <= CTX).mean()), "p50": float(np.median(tot)), "p99": float(np.percentile(tot, 99)), "max": int(tot.max())}
pd.DataFrame(fit).T

## 8 · G2 manual audit file (200 pilot questions × top-3 windows)
Drawn from the **pilot** (dev-only) questions, stratified by relation. The team fills `manual_supports_1..3`.

In [ ]:
pilot = popqa[popqa["is_pilot"]].copy()
quota = (pilot["prop"].value_counts(normalize=True) * 200).apply(np.ceil).astype(int)
aud = pd.concat([g.sample(min(len(g), quota[p]), random_state=7) for p, g in pilot.groupby("prop")])
aud = aud.sample(n=min(200, len(aud)), random_state=7).sort_values(["prop", "id"])
idx = {qid: i for i, qid in enumerate(popqa["id"])}
rows = []
for r in aud.itertuples(index=False):
    i = idx[r.id]; rec = {"id": r.id, "relation": r.prop, "pop_bin": r.pop_bin, "question": r.question,
                         "gold_aliases": " | ".join(r.gold_aliases[:8]), "subject": r.subj}
    for j in range(3):
        d = int(doc_ids[i, j])
        rec[f"window_{j+1}"] = WTEXT.get(d, "")
        rec[f"auto_answer_in_{j+1}"] = int(ans[i, j]); rec[f"auto_subject_in_{j+1}"] = int(subj[i, j])
        rec[f"manual_supports_{j+1}"] = ""
    rows.append(rec)
pd.DataFrame(rows).to_csv(f"{OUT}/audit/g2_support_audit.csv", index=False)
open(f"{OUT}/audit/G2_AUDIT_README.md", "w").write("""# G2 manual audit
For each row and each window_1..3, write in manual_supports_k:
- **1** = the window STATES the fact: subject + relation + one gold answer (e.g. "X was directed by Y").
- **0** = it does not (answer string appears by coincidence, wrong entity with the same name, or fact absent).
Ignore auto_* columns while labelling (they are proxies). Two people label 50 overlapping rows to report agreement.
G2 passes if a useful share of questions has >=1 supporting window in the top-3 (team decides the threshold; report it with a CI).
""")
print(f"audit rows: {len(rows)} | relations: {aud['prop'].value_counts().to_dict()}")

## 9 · Save retrieval cache (+ index)

In [ ]:
pd.DataFrame({"id": popqa["id"], "doc_ids": list(doc_ids), "scores": list(scores),
              "answer_in": list(ans), "subject_in": list(subj), "own_article": list(own)}
            ).to_parquet(f"{OUT}/retrieval/popqa_top{TOPK}.parquet", index=False)
pd.DataFrame({"doc_id": list(WTEXT), "article_idx": [int(ix.doc_article[d]) for d in WTEXT],
              "start_word": [int(ix.doc_start[d]) for d in WTEXT], "title": [WTITLE[d] for d in WTEXT],
              "text": list(WTEXT.values()), "n_tokens": [wt[d] for d in WTEXT]}
            ).to_parquet(f"{OUT}/retrieval/windows_retrieved.parquet", index=False)
if SAVE_INDEX:
    ix.save(f"{OUT}/index"); stage("index saved")
json.dump({"notebook": "02_bm25", "bm25": bcfg.to_dict(), "topk": TOPK, "index_stats": ix.stats,
           "articles": {"path": ART_PATH, "fingerprint": ART_FINGERPRINT, "note": "article_idx = row in articles.parquet"},
           "corpus": CORPUS_INFO,
           "context_fit": fit, "window_tokens_sample": {f"p{q}": float(np.percentile(win_tok, q)) for q in [5, 50, 95, 99]},
           "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT}},
          open(f"{OUT}/meta.json", "w"), indent=2, default=str)
for f in sorted(glob.glob(f"{OUT}/**/*", recursive=True)):
    if os.path.isfile(f): print(f"{os.path.getsize(f)/1e6:10.1f} MB  {os.path.relpath(f, OUT)}")

## 10 · Gate 02 (automatic part of G2)

In [ ]:
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
add("all_questions_have_topk", n_short == 0, f"{n_short} questions with < {TOPK} results", "warn")
add("context_fit_top3_ge_95pct", fit[3]["fit_rate"] >= 0.95, f"{fit[3]['fit_rate']:.1%} fit (p99 {fit[3]['p99']:.0f} tokens)")
add("context_fit_top2_ge_99pct", fit[2]["fit_rate"] >= 0.99, f"{fit[2]['fit_rate']:.1%} fit", "warn")
h3 = float(hit_by_split.loc["val", "answer_hit@3"])
add("val_answer_hit@3_ge_30pct", h3 >= 0.30, f"{h3:.1%} (in_main val)", "warn")
add("indexes_exactly_notebook01_subset",
    len(TITLES) == STATS01["forced_articles"] + STATS01["fill_articles"] and int(ix.doc_article.max()) == len(TITLES) - 1,
    f"{len(TITLES):,} articles = forced {STATS01['forced_articles']:,} + fill {STATS01['fill_articles']:,} (pretraining subset, not full Wikipedia)")
add("window_count_consistent", N_WIN == len(ix.doc_article) == len(ix.doc_start), f"{N_WIN:,}")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"gate": "02_bm25_auto", "status": status, "retrieval_corpus": RETRIEVAL_CORPUS, "manual_audit": "PENDING - label audit/g2_support_audit.csv",
           "checks": checks}, open(f"{OUT}/gate_report.json", "w"), indent=2, default=str)
print(f"\nRetrieval corpus: PRETRAINING SUBSET ({len(TITLES):,} articles, {CORPUS_INFO['share_of_snapshot_articles']:.1%} of the snapshot) - not full Wikipedia")
print(f"GATE 02 (automatic): {status} | G2 manual audit: PENDING | total {(time.time()-T0)/60:.0f} min")
assert not fails, "Automatic retrieval gate failed."

## Next
- Download `audit/g2_support_audit.csv`, label it (README inside), and send the result: that completes **G2**.
- Notebook **03** (pretraining) does not depend on this notebook and can run in parallel on GPU.